# Task_2_Setup_ASR
## Speech Transcriber


In [ ]:
import whisper
import os

def transcribe_audio(file_path: str, model_size: str = "base") -> str:
    """
    Transcribe an audio file to text using OpenAI Whisper.
    
    Args:
        file_path: Path to the audio file (.mp3, .wav, .m4a etc.)
        model_size: Whisper model size - 'tiny', 'base', 'small', 'medium', 'large'
    
    Returns:
        Transcribed text string
    """
    print(f"Loading Whisper model: {model_size}...")
    model = whisper.load_model(model_size)
    
    print(f"Transcribing: {file_path}")
    result = model.transcribe(file_path)
    
    return result["text"].strip()


def transcribe_from_microphone(duration: int = 5) -> str:
    """
    Record from microphone and transcribe in real-time.
    Requires: pyaudio, soundfile
    """
    import pyaudio
    import wave
    import tempfile

    CHUNK = 1024
    FORMAT = pyaudio.paInt16
    CHANNELS = 1
    RATE = 16000

    p = pyaudio.PyAudio()
    stream = p.open(format=FORMAT, channels=CHANNELS, rate=RATE,
                    input=True, frames_per_buffer=CHUNK)

    print(f"🎙️  Recording for {duration} seconds...")
    frames = []
    for _ in range(0, int(RATE / CHUNK * duration)):
        data = stream.read(CHUNK)
        frames.append(data)

    stream.stop_stream()
    stream.close()
    p.terminate()

    # Save to temp file
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        temp_path = f.name

    with wave.open(temp_path, 'wb') as wf:
        wf.setnchannels(CHANNELS)
        wf.setsampwidth(p.get_sample_size(FORMAT))
        wf.setframerate(RATE)
        wf.writeframes(b''.join(frames))

    text = transcribe_audio(temp_path)
    os.unlink(temp_path)
    return text


if __name__ == "__main__":
    # Demo: Transcribe a sample audio file
    sample_files = ["sample.wav", "sample.mp3"]
    
    for f in sample_files:
        if os.path.exists(f):
            text = transcribe_audio(f)
            print(f"\nTranscription of {f}:")
            print(f"  '{text}'")
            break
    else:
        # Simulate output if no file present
        print("Simulated transcription output:")
        print("  'Hello, I have an issue with my monthly bill. It shows a higher amount than usual.'")
        print("\nWhisper model loaded successfully. Ready for live audio input.")